In [1]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols

In [2]:
# ----------------------------
# Helpers
# ----------------------------
def mode_or_nan(s):
    s = s.dropna()
    return s.value_counts().idxmax() if not s.empty else np.nan

def safe_div(a, b):
    b = b.replace(0, np.nan)
    return a / b

In [11]:
# ----------------------------
# 1) Load data
# ----------------------------
attendance = pd.read_csv("data/attendance.csv")
entity_schedule = pd.read_csv("data/entity_schedule.csv")
link = pd.read_csv("data/link_attraction_park.csv")
waiting = pd.read_csv("data/waiting_times.csv")
weather = pd.read_csv("data/weather_data.csv")

# ----------------------------
# 2–3) Filter attractions to PortAventura World via link_attraction_park.csv
#     link file has a single column like "ATTRACTION;PARK"
# ----------------------------
link_col = link.columns[0]
link[["ATTRACTION", "PARK"]] = link[link_col].astype(str).str.split(";", n=1, expand=True)
link["ATTRACTION"] = link["ATTRACTION"].str.strip()
link["PARK"] = link["PARK"].str.strip()

pa_attractions = set(link.loc[link["PARK"] == "PortAventura World", "ATTRACTION"])

# normalize attraction names in waiting_times + schedule
waiting["ENTITY_DESCRIPTION_SHORT"] = waiting["ENTITY_DESCRIPTION_SHORT"].astype(str).str.strip()
entity_schedule["ENTITY_DESCRIPTION_SHORT"] = entity_schedule["ENTITY_DESCRIPTION_SHORT"].astype(str).str.strip()

waiting = waiting[waiting["ENTITY_DESCRIPTION_SHORT"].isin(pa_attractions)].copy()


In [12]:
# ----------------------------
# 1) Target: daily avg WAIT_TIME_MAX per attraction
# IMPORTANT: compute average only when attraction is open (OPEN_TIME > 0)
# to avoid "closed intervals" forcing wait=0 into the average.
# ----------------------------
waiting["WORK_DATE"] = pd.to_datetime(waiting["WORK_DATE"], errors="coerce")
waiting = waiting.dropna(subset=["WORK_DATE"])
waiting["date"] = waiting["WORK_DATE"].dt.floor("D")

waiting_open = waiting[waiting["OPEN_TIME"] > 0].copy()

daily_attr = (
    waiting_open
    .groupby(["date", "ENTITY_DESCRIPTION_SHORT"], as_index=False)
    .agg(
        wait_time_avg=("WAIT_TIME_MAX", "mean"),
        guests_sum=("GUEST_CARRIED", "sum"),
        adjcap_sum=("ADJUST_CAPACITY", "sum"),
        open_min_sum=("OPEN_TIME", "sum"),
        up_min_sum=("UP_TIME", "sum"),
        nb_units_med=("NB_UNITS", "median"),
        nb_max_unit=("NB_MAX_UNIT", "max"),
    )
)

# Derived “mechanism” features (avoid multicollinearity from raw components)
daily_attr["utilization"] = safe_div(daily_attr["guests_sum"], daily_attr["adjcap_sum"])
daily_attr["availability"] = safe_div(daily_attr["up_min_sum"], daily_attr["open_min_sum"])
daily_attr["eff_cap"] = daily_attr["adjcap_sum"] * daily_attr["availability"]

# Basic calendar controls
daily_attr["dow"] = daily_attr["date"].dt.dayofweek  # 0=Mon
daily_attr["month"] = daily_attr["date"].dt.month

# ----------------------------
# 3) Attendance: keep only PortAventura World, merge on date
# ----------------------------
attendance["USAGE_DATE"] = pd.to_datetime(attendance["USAGE_DATE"], errors="coerce")
attendance = attendance.dropna(subset=["USAGE_DATE"])
attendance["date"] = attendance["USAGE_DATE"].dt.floor("D")

attendance_pa = attendance.loc[
    attendance["FACILITY_NAME"] == "PortAventura World",
    ["date", "attendance"]
].copy()

daily = daily_attr.merge(attendance_pa, on="date", how="left")

# ----------------------------
# 2) entity_schedule: compute scheduled open minutes for attractions, merge
# (use latest UPDATE_TIME per attraction-day if multiple rows exist)
# ----------------------------
entity_schedule["WORK_DATE"] = pd.to_datetime(entity_schedule["WORK_DATE"], errors="coerce")
entity_schedule["DEB_TIME"] = pd.to_datetime(entity_schedule["DEB_TIME"], errors="coerce")
entity_schedule["FIN_TIME"] = pd.to_datetime(entity_schedule["FIN_TIME"], errors="coerce")
entity_schedule["UPDATE_TIME"] = pd.to_datetime(entity_schedule["UPDATE_TIME"], errors="coerce")

es = entity_schedule.copy()
es = es[es["ENTITY_TYPE"].eq("ATTR")]
es = es[es["ENTITY_DESCRIPTION_SHORT"].isin(pa_attractions)]
es = es.dropna(subset=["WORK_DATE", "DEB_TIME", "FIN_TIME"])
es["date"] = es["WORK_DATE"].dt.floor("D")

# keep latest update per date+attraction
es = es.sort_values("UPDATE_TIME").groupby(["date", "ENTITY_DESCRIPTION_SHORT"], as_index=False).tail(1)

es["scheduled_open_min"] = (es["FIN_TIME"] - es["DEB_TIME"]).dt.total_seconds() / 60.0
es = es[["date", "ENTITY_DESCRIPTION_SHORT", "scheduled_open_min"]]

daily = daily.merge(es, on=["date", "ENTITY_DESCRIPTION_SHORT"], how="left")

# ----------------------------
# 4) Weather: hourly -> daily aggregates
#     - numeric: median per day
#     - categorical: mode per day
#     Robust dt_iso parsing for strings like "1999-01-01 00:00:00 +0000 UTC"
# ----------------------------
weather = weather.copy()

# 1) Parse dt_iso robustly
if "dt_iso" in weather.columns:
    # Remove trailing " UTC" so %z works
    dt_clean = weather["dt_iso"].astype(str).str.replace(" UTC", "", regex=False)

    weather["dt_iso_parsed"] = pd.to_datetime(
        dt_clean,
        format="%Y-%m-%d %H:%M:%S %z",
        errors="coerce"
    )
else:
    raise ValueError("weather_data.csv must contain dt_iso column")

weather = weather.dropna(subset=["dt_iso_parsed"])

# 2) Make daily key timezone-naive for clean merge with your other data
weather["date"] = weather["dt_iso_parsed"].dt.tz_convert(None).dt.floor("D")

# Make sure daily['date'] is also timezone-naive
daily["date"] = pd.to_datetime(daily["date"], errors="coerce").dt.floor("D")

# 3) Choose columns that exist
numeric_candidates = [
    "temp", "feels_like", "dew_point", "pressure", "humidity",
    "wind_speed", "wind_deg", "wind_gust", "clouds_all",
    "rain_1h", "rain_3h", "snow_1h", "snow_3h", "visibility"
]
cat_candidates = ["weather_main", "weather_description", "weather_icon", "weather_id"]

num_cols = [c for c in numeric_candidates if c in weather.columns]
cat_cols = [c for c in cat_candidates if c in weather.columns]

# 4) Aggregate daily
daily_weather = weather[["date"]].drop_duplicates().sort_values("date").copy()

if num_cols:
    daily_weather_num = weather.groupby("date", as_index=False)[num_cols].median()
    daily_weather = daily_weather.merge(daily_weather_num, on="date", how="left")

if cat_cols:
    daily_weather_cat = weather.groupby("date", as_index=False)[cat_cols].agg(mode_or_nan)
    daily_weather = daily_weather.merge(daily_weather_cat, on="date", how="left")

# 5) Merge into main daily dataset
daily = daily.merge(daily_weather, on="date", how="left")

# 6) (Recommended) Fill missing numeric weather so OLS doesn't drop most rows
#    This is especially useful if weather coverage doesn't fully overlap your park dates.
for c in num_cols:
    daily[c] = daily[c].fillna(daily[c].median())


In [13]:
# ----------------------------
# 5) COVID dummies (Spain state of alarm started 2020-03-14; second ended 2021-05-09)
# ----------------------------
covid_start = pd.Timestamp("2020-03-14")
covid_end = pd.Timestamp("2021-05-09")

daily["covid"] = ((daily["date"] >= covid_start) & (daily["date"] <= covid_end)).astype(int)
daily["post_covid"] = (daily["date"] > covid_end).astype(int)
# pre-covid is the baseline (both dummies = 0)

# ----------------------------
# 6) Season dummies (4 seasons)
# ----------------------------
def month_to_season(m):
    if m in (12, 1, 2):
        return "winter"
    if m in (3, 4, 5):
        return "spring"
    if m in (6, 7, 8):
        return "summer"
    return "autumn"

daily["season"] = daily["month"].apply(month_to_season)

# ----------------------------
# 7) Resulting daily-based dataframe
# ----------------------------
# (Optional) drop rows with missing key drivers
model_df = daily.copy()
model_df = model_df.dropna(subset=["wait_time_avg", "utilization", "availability"])


In [15]:
import numpy as np
import patsy
import statsmodels.formula.api as smf

# ----------------------------
# 8) Linear regression (robust version)
#   - attraction FE: C(ENTITY_DESCRIPTION_SHORT)
#   - DOW FE: C(dow)
#   - season FE: C(season)
#   - COVID interactions with utilization
#   - Clustered SEs by attraction, aligned to used rows
# ----------------------------

# (Optional but recommended) fill some common missing merges so you don't lose lots of rows
if "attendance" in model_df.columns:
    model_df["attendance"] = model_df["attendance"].fillna(model_df["attendance"].median())
if "scheduled_open_min" in model_df.columns:
    model_df["scheduled_open_min"] = model_df["scheduled_open_min"].fillna(model_df["scheduled_open_min"].median())

# Weather terms (only include if column exists AND has at least some non-missing values)
weather_candidates = ["temp", "rain_1h", "wind_speed", "clouds_all", "humidity"]
weather_terms = [
    c for c in weather_candidates
    if c in model_df.columns and model_df[c].notna().any()
]

# Build formula pieces
base_terms = [
    "utilization",
    "availability",
    "np.log1p(attendance)",
    "scheduled_open_min",
    "nb_units_med",
    "covid",
    "post_covid",
    "utilization:covid",
    "utilization:post_covid",
    "C(dow)",
    "C(season)",
    "C(ENTITY_DESCRIPTION_SHORT)"
]

all_terms = base_terms + weather_terms
formula = "wait_time_avg ~ " + " + ".join(all_terms)

# Build design matrices to get the exact rows statsmodels will use (after dropping missing)
y, X = patsy.dmatrices(formula, data=model_df, return_type="dataframe")
used_idx = X.index

print(f"Rows in model_df: {len(model_df):,}")
print(f"Rows used in regression (after NA handling): {len(used_idx):,}")
print(f"Weather terms included: {weather_terms}")

# Align clustering groups to the used rows
groups = model_df.loc[used_idx, "ENTITY_DESCRIPTION_SHORT"]

# Fit OLS with clustered SEs
fit = smf.ols(formula, data=model_df.loc[used_idx]).fit(
    cov_type="cluster",
    cov_kwds={"groups": groups}
)

print(fit.summary())


c:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Rows in model_df: 29,847
Rows used in regression (after NA handling): 29,253
Weather terms included: ['temp', 'rain_1h', 'wind_speed', 'clouds_all', 'humidity']
                            OLS Regression Results                            
Dep. Variable:          wait_time_avg   R-squared:                       0.719
Model:                            OLS   Adj. R-squared:                  0.719
Method:                 Least Squares   F-statistic:                 4.230e+08
Date:                Mon, 09 Feb 2026   Prob (F-statistic):          5.34e-102
Time:                        15:55:21   Log-Likelihood:            -1.0880e+05
No. Observations:               29253   AIC:                         2.177e+05
Df Residuals:                   29204   BIC:                         2.181e+05
Df Model:                          48                                         
Covariance Type:              cluster                                         
                                                 

c:\Users\artur\Desktop\ESSEC MS IN DATA SCIENCE\0 STUDY\M2\Hackathon\code\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 48, but rank is 23
  warnings.warn('covariance of constraints does not have full '
